# Test the model on real voice notes

Transcribes the audio files in `real-data/` (WhatsApp `.opus` voice notes, or `.wav`, `.mp3`, `.m4a`, `.ogg`)
with `dyeto/kanbior-fon-asr-v1-ct2` (faster-whisper, CPU int8). Files are decoded and resampled to
16 kHz mono by faster-whisper (PyAV), so no conversion is needed.


In [11]:
from pathlib import Path

from faster_whisper import WhisperModel
from faster_whisper.audio import decode_audio
from IPython.display import Audio, display

MODEL = "dyeto/kanbior-fon-asr-v1-ct2"   # or a local folder, e.g. "runs/kanbior-fon-asr-v1-step12000"
AUDIO_DIR = Path("real-data")
VAD = True    # skip silence and background noise before transcribing (helps with voice notes)

model = WhisperModel(MODEL, device="cpu", compute_type="int8")
files = sorted(p for p in AUDIO_DIR.iterdir() if p.suffix.lower() in {".opus", ".ogg", ".wav", ".mp3", ".m4a"})
print(f"{len(files)} files in {AUDIO_DIR}/")


7 files in real-data/


[2026-11-23 22:44:34.375] [ctranslate2] [thread 597659] [warning] Tip: experimental MPS backend is available and often 2x+ faster than CPU on Apple Silicon (varies by workload and hardware). Try device="mps".


In [2]:
def transcribe(path, vad=VAD):
    audio = decode_audio(str(path), sampling_rate=16000)   # float32 mono, 16 kHz
    segments, _ = model.transcribe(audio, language="fr", beam_size=1, without_timestamps=True, vad_filter=vad)
    return audio, " ".join(s.text.strip() for s in segments)


for path in files:
    audio, text = transcribe(path)
    print(f"{path.name}  ({len(audio) / 16000:.1f} s)")
    display(Audio(audio, rate=16000))
    print(f"  {text}\n")


WhatsAppAudio1.opus  (23.1 s)


  Hwɛ ɔ́, dǒ tɛ́ mɛ aga se ɖɔ yě nɔ́ ɖɔ Paincure Natué. Natupe Nucure yě nɔ́ ɖɔ ǎ, Natupe Nucure yě nɔ́ ɖɔ atlito káká. Natupe Nucure yě nɔ́ ɖɔ yě kún nɔ́ ɖɔ Paincure Natué ó.

WhatsAppAudio2.opus  (19.6 s)


  Mi mɛ ɖaxó to lɛ́ɛ ɖo xó ɖɔ wɛ ɔ́, a ɖǒ ná nɔ se bo nɔ nɔ sin mɛ bo nɔ se mɛ ɖaxó lɛ́ɛ sín xó. Yě lɛ́ɛ ɖo yě mɔ gbɛ nukɔn bo ɖo gbɛdlɛ́ wɛ. Mi gbɔn ɔ́, mi mɔ̌ ɖo ní mɛ ɔ́, mi mɔ̌ ɖaxó lɛ́ɛ sín xó.

WhatsAppAudio3.opus  (17.1 s)


  Akwínnɔ nǔ e è nɔ ylɔ́ ɖɔ telobɔ́ ɔ́ a? Nyɔ̌nu ɖu telobɔ́ din ɔ́ é sɔ́ n'i mɛ, é ní jí. A ɖǒ telobɔ́ ɔ́, un hɛn ná bó ná akwǎ. A ɖǒ telobɔ́ nú mì ɖu telobɔ́ ɔ́ ganjí.

WhatsAppAudio4.opus  (8.4 s)


  Mɛɖaxó, a ka tunwun ɔ́ é má nyí jɛ kpikpa zé ɖó ayǐ élɔ́ ɔ́ a? Yě nɔ́ ylɔ́ ɖɔ Mpí ɔ́ a tunwun a?



One file, with and without VAD (if a transcript has missing words or invented text, compare the two):

In [4]:
path = files[0]
print("VAD on :", transcribe(path, vad=True)[1])
print("VAD off:", transcribe(path, vad=False)[1])


VAD on : Hwɛ ɔ́, dǒ tɛ́ mɛ aga se ɖɔ yě nɔ́ ɖɔ Paincure Natué. Natupe Nucure yě nɔ́ ɖɔ ǎ, Natupe Nucure yě nɔ́ ɖɔ atlito káká. Natupe Nucure yě nɔ́ ɖɔ yě kún nɔ́ ɖɔ Paincure Natué ó.
VAD off: Hwɛ ɔ́, dǒ pɛ́ mɛ aga se ɖɔ yě nɔ́ ɖɔ Paincure Natuí. Natuí Paincure yě nɔ́ ɖɔ ǎ, Natuí Paincure yě nɔ́ ɖɔ, ǎ, ǎ, ǎ, ɖitó káká, hwlɛ́wlɛ́ ɖitó káká, Natuí Paincure yě nɔ́ ɖɔ, yě kɔ́n nɔ ɖɔ Paincure Natuí ó.


In [ ]:
path = files[5]
path

PosixPath('real-data/WhatsAppAudio5.opus')

In [8]:
print("VAD on :", transcribe(path, vad=True)[1])
print("VAD off:", transcribe(path, vad=False)[1])

VAD on : Étɛ́ nɔví towe bló nú we?
VAD off: Étɛ́ nɔví towe bló nú we?


In [10]:
path = files[5]
print("VAD on :", transcribe(path, vad=True)[1])
print("VAD off:", transcribe(path, vad=False)[1])

VAD on : A blɔ ɖe xwe Benɛ́ɛ tɔn ɔ́, hwetɛ́nu wɛ?
VAD off: A blɔ ɖe xwe Benɛ́ɛ tɔn ɔ́, hwetɛ́nu wɛ.


In [12]:
path = files[6]
print("VAD on :", transcribe(path, vad=True)[1])
print("VAD off:", transcribe(path, vad=False)[1])

VAD on : Un gosín fínɛ́ ɔ́ é lín kpɛɖé, un si lɛ́kɔ wá ɔ́, mɛ e jɛ sɔ un si wá.
VAD off: Un gosín fínɛ́ ɔ́, é lín kpɛɖé. Un si lɛ́kɔ wá ɔ́, un mɛ e jɛ sɔ ɔ́, un si wá.
